# Homework

In [2]:
import pickle
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor
import seaborn as sns
import matplotlib.pyplot as plt
import mlflow

KeyboardInterrupt: 

In [ ]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("duration-prediction")

<Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1747987569201, experiment_id='1', last_update_time=1747987569201, lifecycle_stage='active', name='duration-prediction', tags={}>

In [ ]:
def clean(df):
  # Duration in minutes

  df['duration'] = (df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']) / pd.Timedelta(minutes=1)

  # Remove outliers out of bounds 1 - 60 minutes

  df = df[(df['duration'] >= 1) & (df['duration'] <= 60)]

  return df


### Downloading the data

In [ ]:
df_jan = pd.read_parquet('green_tripdata_2023-01.parquet')
df_jan = clean(df_jan)

### One-hot encoding

In [ ]:
def dict_extract(df):
  categorical = ['PULocationID', 'DOLocationID']
  df[categorical] = df[categorical].astype(str)
  df_dict = df[categorical].to_dict(orient='records')

  return df_dict

dict_jan = dict_extract(df_jan)

dv = DictVectorizer()
X_train = dv.fit_transform(dict_jan)

### Training a model

In [ ]:
y_train = df_jan['duration'].values
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_train)

In [ ]:
df_val = pd.read_parquet('green_tripdata_2023-02.parquet')

In [ ]:
df_val = clean(df_val)

dict_val = dict_extract(df_val)
X_val = dv.transform(dict_val)

y_val = df_val['duration'].values

In [ ]:
y_pred = lr.predict(X_val)
print('RMSE on validation data:', root_mean_squared_error(y_val, y_pred))

RMSE on validation data: 7.356158578732979


In [ ]:
with mlflow.start_run() as run:
    # mlflow.set_tag("developer", "elvis")
    # mlflow.set_tag("model", "Lasso")
    mlflow.autolog()
    
    alpha = 0.1
    
    # mlflow.log_param("alpha", alpha)
    lr = Lasso(alpha)
    lr.fit(X_train, y_train)
    
    y_pred = lr.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    # mlflow.log_metric("rmse", rmse)
    
    

2025/05/25 04:34:21 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


In [ ]:
with open('models/lin_regression-1.bin', 'wb') as f:
  pickle.dump((dv, lr), f)

In [ ]:
def load_pickle(filename: str):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)

In [ ]:
X_train, y_train = load_pickle("output/train.pkl")
X_val, y_val = load_pickle("output/val.pkl")

In [ ]:

mlflow.sklearn.autolog(log_datasets=False)
with mlflow.start_run() as run:
    # mlflow.set_tag("developer", "elvis")
    # mlflow.set_tag("model", "Lasso")
    
    rf = RandomForestRegressor(max_depth=10, random_state=0)

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    
    rmse = root_mean_squared_error(y_val, y_pred)
    # mlflow.log_metric("rmse", rmse)

In [4]:
import os
import pickle
import mlflow
import numpy as np
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from hyperopt.pyll import scope
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("random-forest-hyperopt")


def load_pickle(filename: str):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)


def run_optimization(num_trials: int):

    X_train, y_train = load_pickle("output/train.pkl")
    X_val, y_val = load_pickle("output/val.pkl")

    def objective(params):
        with mlflow.start_run():
            mlflow.set_tag("model", "RandomForestRegressor")
            mlflow.set_tag("developer", "elvis")
            mlflow.log_params(params)
            rf = RandomForestRegressor(**params)
            rf.fit(X_train, y_train)
            y_pred = rf.predict(X_val)
            rmse = root_mean_squared_error(y_val, y_pred)
            
            mlflow.log_metric("rmse", rmse)

        return {'loss': rmse, 'status': STATUS_OK}

    search_space = {
        'max_depth': scope.int(hp.quniform('max_depth', 1, 20, 1)),
        'n_estimators': scope.int(hp.quniform('n_estimators', 10, 50, 1)),
        'min_samples_split': scope.int(hp.quniform('min_samples_split', 2, 10, 1)),
        'min_samples_leaf': scope.int(hp.quniform('min_samples_leaf', 1, 4, 1)),
        'random_state': 42
    }

    rstate = np.random.default_rng(4)  # for reproducible results
    best_results = fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,
        max_evals=num_trials,
        trials=Trials(),
        rstate=rstate
    )
    
    print("Best hyperparameters:", best_results)


if __name__ == '__main__':
    run_optimization(50)


  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

🏃 View run bemused-cod-425 at: http://127.0.0.1:5000/#/experiments/3/runs/ec349c91d57d4a1a9f6b6065d06db472

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3

🏃 View run chill-carp-337 at: http://127.0.0.1:5000/#/experiments/3/runs/41af940d96b44dc68e6d9bf65b14a51b

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3                   

🏃 View run omniscient-moose-223 at: http://127.0.0.1:5000/#/experiments/3/runs/c9eb958c3b0545d78dfc70e9f573af39

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3                   

🏃 View run efficient-cod-790 at: http://127.0.0.1:5000/#/experiments/3/runs/9235680201534254b071acbe9777c757

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3                   

🏃 View run salty-kit-372 at: http://127.0.0.1:5000/#/experiments/3/runs/9d360731577e45bf993177dc4d25aee4

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3                   

🏃 View run colorful-fawn-922 at: http://127.0.0.1:5000/#/experiments/3/runs

In [10]:
client = MlflowClient(tracking_uri="http://127.0.0.1:5000")


In [13]:
runs

[<Run: data=<RunData: metrics={'rmse': 5.31366503340177}, params={'max_depth': '20',
  'min_samples_leaf': '2',
  'min_samples_split': '9',
  'n_estimators': '21',
  'random_state': '42'}, tags={'developer': 'elvis',
  'mlflow.runName': 'mercurial-shad-226',
  'mlflow.source.name': '/home/codespace/.local/lib/python3.12/site-packages/ipykernel_launcher.py',
  'mlflow.source.type': 'LOCAL',
  'mlflow.user': 'codespace',
  'model': 'RandomForestRegressor'}>, info=<RunInfo: artifact_uri='mlflow-artifacts:/3/821c3ff8ebf8408f8fdb0016b8929f1b/artifacts', end_time=1748253890769, experiment_id='3', lifecycle_stage='active', run_id='821c3ff8ebf8408f8fdb0016b8929f1b', run_name='mercurial-shad-226', run_uuid='821c3ff8ebf8408f8fdb0016b8929f1b', start_time=1748253883858, status='FINISHED', user_id='codespace'>, inputs=<RunInputs: dataset_inputs=[]>>]

In [12]:
client.search_experiments()

[<Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/artifacts/4', creation_time=1748254026866, experiment_id='4', last_update_time=1748254026866, lifecycle_stage='active', name='random-forest-best-models', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1748212229695, experiment_id='3', last_update_time=1748212229695, lifecycle_stage='active', name='random-forest-hyperopt', tags={}>,
 <Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/2', creation_time=1748125387497, experiment_id='2', last_update_time=1748125387497, lifecycle_stage='active', name='duration-prediction-training', tags={}>,
 <Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1747987569201, experiment_id='1', last_update_time=1747987569201, lifecycle_stage='active', name='duration-prediction', tags={}>,
 <Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-exp